# TP1bis — Introduction à GeoPandas — Corrigé

Ce corrigé couvre toutes les parties du TP GeoPandas avec des explications détaillées.

## Partie 1 — Premiers pas avec GeoPandas

In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from shapely.geometry import Point, LineString, Polygon

# 1. Chargement du dataset
world = gpd.read_file(gpd.datasets.get_path("naturalearth_lowres"))

print("=== 5 premières lignes ===")
display(world.head())

In [ ]:
# 2. Exploration des colonnes
print("Colonnes:", list(world.columns))
print(f"\nNombre de pays: {len(world)}")
print(f"\nTypes de données:\n{world.dtypes}")

In [ ]:
# 3. Type de géométrie
print("=== Types de géométrie ===")
print(world.geometry.type.value_counts())

In [ ]:
# 4. Système de coordonnées
print("CRS (Coordinate Reference System):")
print(world.crs)

In [ ]:
# 5. Carte simple
fig, ax = plt.subplots(figsize=(15, 8))
world.plot(ax=ax, color="lightgreen", edgecolor="black", linewidth=0.5)
ax.set_title("Carte du monde — Natural Earth", fontsize=14)
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
plt.tight_layout()
plt.show()

## Partie 2 — Exploration et sélections

In [ ]:
# 1. Pays d'Europe
europe = world[world["continent"] == "Europe"]
print(f"Nombre de pays en Europe: {len(europe)}")
display(europe[["name", "pop_est", "gdp_md_est"]].head(10))

In [ ]:
# 2. Pays avec population > 50 millions
big_countries = world[world["pop_est"] > 50_000_000]
print(f"Pays avec plus de 50M d'habitants: {len(big_countries)}")
display(big_countries[["name", "pop_est", "continent"]].sort_values("pop_est", ascending=False))

In [ ]:
# 3. La France
france = world[world["name"] == "France"]
print("=== La France ===")
display(france)

# Afficher la géométrie
fig, ax = plt.subplots(figsize=(8, 8))
france.plot(ax=ax, color="#3498db", edgecolor="black")
ax.set_title("France métropolitaine")
plt.tight_layout()
plt.show()

In [ ]:
# 4. Aire de chaque pays (attention: en degrés carrés !)
world["area_deg2"] = world.geometry.area
print("=== Aires en degrés carrés (peu significatif) ===")
display(world[["name", "area_deg2"]].sort_values("area_deg2", ascending=False).head(10))

In [ ]:
# 5. Tri par superficie
print("=== Top 10 pays par superficie ===")
display(world.sort_values("area_deg2", ascending=False)[["name", "continent", "area_deg2"]].head(10))

## Partie 3 — Systèmes de coordonnées (CRS)

In [ ]:
# 1. CRS actuel
print("CRS actuel:")
print(f"  - Code: {world.crs}")
print(f"  - Nom: {world.crs.name}")
print(f"  - Unité: degrés (système géographique)")

In [ ]:
# 2. Reprojection en Lambert-93 (France)
france_lambert = france.to_crs(epsg=2154)
print("France en Lambert-93:")
print(f"  - CRS: {france_lambert.crs}")

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

france.plot(ax=axes[0], color="#3498db", edgecolor="black")
axes[0].set_title("WGS84 (EPSG:4326)")

france_lambert.plot(ax=axes[1], color="#e74c3c", edgecolor="black")
axes[1].set_title("Lambert-93 (EPSG:2154)")

plt.tight_layout()
plt.show()

In [ ]:
# 3. Reprojection en Equal Area pour les calculs d'aire
world_equal_area = world.to_crs(epsg=6933)  # World Cylindrical Equal Area

# 4. Comparaison des aires
world["area_km2"] = world_equal_area.geometry.area / 1e6  # m² -> km²

print("=== Comparaison aires (degrés² vs km²) ===")
comparison = world[["name", "area_deg2", "area_km2"]].sort_values("area_km2", ascending=False).head(10)
comparison["area_km2"] = comparison["area_km2"].apply(lambda x: f"{x:,.0f}")
display(comparison)

In [ ]:
# 5. Différence to_crs vs set_crs
print("=== to_crs() vs set_crs() ===")
print()
print("to_crs(): TRANSFORME les coordonnées vers le nouveau CRS")
print("          → Les géométries sont recalculées")
print()
print("set_crs(): DÉFINIT le CRS sans transformer les coordonnées")
print("          → À utiliser quand les données n'ont pas de CRS défini")
print("          → ATTENTION: peut produire des résultats incorrects si mal utilisé")

## Partie 4 — Création de géométries

In [ ]:
# 1. Création d'un GeoDataFrame avec 5 villes françaises
villes_data = {
    "ville": ["Paris", "Lyon", "Marseille", "Toulouse", "Bordeaux"],
    "longitude": [2.3522, 4.8357, 5.3698, 1.4442, -0.5792],
    "latitude": [48.8566, 45.7640, 43.2965, 43.6047, 44.8378],
    "population": [2161000, 516092, 870731, 479553, 257804]
}

# Création des points
geometry = [Point(lon, lat) for lon, lat in zip(villes_data["longitude"], villes_data["latitude"])]

# 2. GeoDataFrame avec CRS WGS84
villes = gpd.GeoDataFrame(villes_data, geometry=geometry, crs="EPSG:4326")

print("=== GeoDataFrame des villes ===")
display(villes)

In [ ]:
# 3. Tracer les villes sur une carte de France
fig, ax = plt.subplots(figsize=(10, 10))

# Fond: France
france.plot(ax=ax, color="#ecf0f1", edgecolor="black")

# Villes
villes.plot(ax=ax, color="#e74c3c", markersize=100, zorder=5)

# Annotations
for idx, row in villes.iterrows():
    ax.annotate(row["ville"], xy=(row.geometry.x, row.geometry.y),
                xytext=(5, 5), textcoords="offset points", fontsize=10, fontweight="bold")

ax.set_title("Principales villes de France", fontsize=14)
ax.set_xlim(-5, 10)
ax.set_ylim(41, 52)
plt.tight_layout()
plt.show()

In [ ]:
# 4-5. Buffers de 50 km
# D'abord reprojeter en mètres (Lambert-93)
villes_lambert = villes.to_crs(epsg=2154)

# Créer les buffers (50 km = 50000 m)
villes_lambert["buffer_50km"] = villes_lambert.geometry.buffer(50000)

# Créer un GeoDataFrame avec les buffers
buffers = villes_lambert.set_geometry("buffer_50km")[["ville", "buffer_50km"]]
buffers = buffers.rename(columns={"buffer_50km": "geometry"}).set_geometry("geometry")

# Reprojeter en WGS84 pour l'affichage
buffers_wgs84 = buffers.to_crs(epsg=4326)

# Visualisation
fig, ax = plt.subplots(figsize=(10, 10))

france.plot(ax=ax, color="#ecf0f1", edgecolor="black")
buffers_wgs84.plot(ax=ax, color="#3498db", alpha=0.3, edgecolor="#2980b9", linewidth=2)
villes.plot(ax=ax, color="#e74c3c", markersize=100, zorder=5)

for idx, row in villes.iterrows():
    ax.annotate(row["ville"], xy=(row.geometry.x, row.geometry.y),
                xytext=(5, 5), textcoords="offset points", fontsize=10, fontweight="bold")

ax.set_title("Villes avec zones tampons de 50 km", fontsize=14)
ax.set_xlim(-5, 10)
ax.set_ylim(41, 52)
plt.tight_layout()
plt.show()

## Partie 5 — Opérations spatiales

In [ ]:
# 1. Intersection avec l'équateur
equateur = LineString([(-180, 0), (180, 0)])
equateur_gdf = gpd.GeoDataFrame({"name": ["Équateur"]}, geometry=[equateur], crs="EPSG:4326")

# Pays traversés par l'équateur
pays_equateur = world[world.intersects(equateur)]
print(f"=== Pays traversés par l'équateur ({len(pays_equateur)}) ===")
print(list(pays_equateur["name"]))

In [ ]:
# 2. Contains/Within
paris_point = Point(2.3522, 48.8566)

# Vérifier dans quel pays se trouve Paris
for idx, row in world.iterrows():
    if row.geometry.contains(paris_point):
        print(f"Paris est en: {row['name']}")
        break

# Alternative avec within
print(f"\nParis est dans la France: {paris_point.within(france.geometry.values[0])}")

In [ ]:
# 3. Distance entre Paris et les autres villes
# Reprojeter en Lambert-93 pour avoir des distances en mètres
villes_lambert = villes.to_crs(epsg=2154)
paris_lambert = villes_lambert[villes_lambert["ville"] == "Paris"].geometry.values[0]

villes_lambert["distance_paris_km"] = villes_lambert.geometry.distance(paris_lambert) / 1000

print("=== Distance depuis Paris ===")
display(villes_lambert[["ville", "distance_paris_km"]].sort_values("distance_paris_km"))

In [ ]:
# 4. Centroïdes des pays européens
europe_copy = europe.copy()
europe_copy["centroid"] = europe_copy.geometry.centroid

# Visualisation
fig, ax = plt.subplots(figsize=(12, 10))

europe.plot(ax=ax, color="#ecf0f1", edgecolor="black")
centroids_gdf = gpd.GeoDataFrame(europe_copy, geometry="centroid")
centroids_gdf.plot(ax=ax, color="#e74c3c", markersize=50)

ax.set_title("Centroïdes des pays européens", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# 5. Union des pays de l'UE (simplification: quelques pays)
eu_countries = ["France", "Germany", "Italy", "Spain", "Poland", "Netherlands", "Belgium"]
eu = world[world["name"].isin(eu_countries)]

# Union de toutes les géométries
eu_union = eu.unary_union

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

eu.plot(ax=axes[0], cmap="tab10", edgecolor="black")
axes[0].set_title("Pays séparés")

gpd.GeoDataFrame({"name": ["EU"]}, geometry=[eu_union], crs=eu.crs).plot(
    ax=axes[1], color="#3498db", edgecolor="black"
)
axes[1].set_title("Union (un seul polygone)")

plt.tight_layout()
plt.show()

## Partie 6 — Jointures spatiales

In [ ]:
# 1. Chargement des villes
cities = gpd.read_file(gpd.datasets.get_path("naturalearth_cities"))
print(f"Nombre de villes: {len(cities)}")
display(cities.head())

In [ ]:
# 2. Jointure spatiale: associer chaque ville à son pays
cities_with_country = gpd.sjoin(cities, world[["name", "continent", "geometry"]], 
                                 how="left", predicate="within")

cities_with_country = cities_with_country.rename(columns={"name_left": "city", "name_right": "country"})

print("=== Villes avec leur pays ===")
display(cities_with_country[["city", "country", "continent"]].head(15))

In [ ]:
# 3. Nombre de villes par pays
cities_per_country = cities_with_country.groupby("country").size().sort_values(ascending=False)
print("=== Nombre de villes par pays (top 15) ===")
print(cities_per_country.head(15))

In [ ]:
# 4. Pays sans ville dans le dataset
countries_with_cities = set(cities_with_country["country"].dropna())
all_countries = set(world["name"])
countries_without_cities = all_countries - countries_with_cities

print(f"=== Pays sans ville dans le dataset ({len(countries_without_cities)}) ===")
print(sorted(countries_without_cities)[:20], "...")

## Partie 7 — Lecture/écriture de fichiers

In [ ]:
# 1. Export GeoJSON
europe.to_file("europe.geojson", driver="GeoJSON")
print("✓ europe.geojson créé")

In [ ]:
# 2. Export Shapefile
europe.to_file("europe_shp/europe.shp")
print("✓ Shapefile créé")

# Lister les fichiers créés
import os
print("\nFichiers créés:")
for f in os.listdir("europe_shp"):
    print(f"  - {f}")

In [ ]:
# 3. Export GeoPackage (format recommandé)
europe.to_file("europe.gpkg", driver="GPKG")
print("✓ europe.gpkg créé")

In [ ]:
# 4. Rechargement et vérification
europe_reload = gpd.read_file("europe.gpkg")

print("=== Vérification après rechargement ===")
print(f"Même nombre de lignes: {len(europe) == len(europe_reload)}")
print(f"Mêmes colonnes: {list(europe.columns) == list(europe_reload.columns)}")
print(f"Même CRS: {europe.crs == europe_reload.crs}")

## Partie 8 — Cartographie thématique

In [ ]:
# 8.1 Carte choroplèthe
fig, ax = plt.subplots(figsize=(15, 8))

world.plot(column="pop_est", ax=ax, legend=True, 
           legend_kwds={"label": "Population", "shrink": 0.6},
           cmap="YlOrRd", edgecolor="black", linewidth=0.3)

ax.set_title("Population mondiale par pays", fontsize=14)
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
plt.tight_layout()
plt.show()

In [ ]:
# 8.2 Carte par catégories (continents)
fig, ax = plt.subplots(figsize=(15, 8))

world.plot(column="continent", ax=ax, legend=True,
           legend_kwds={"loc": "lower left"},
           cmap="Set2", edgecolor="black", linewidth=0.3)

ax.set_title("Carte du monde par continent", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# 8.3 Carte multi-couches
fig, ax = plt.subplots(figsize=(12, 10))

# Couche 1: France
france.plot(ax=ax, color="#bdc3c7", edgecolor="#2c3e50", linewidth=2)

# Couche 2: Buffers
buffers_wgs84.plot(ax=ax, color="#3498db", alpha=0.3, edgecolor="#2980b9", linewidth=1)

# Couche 3: Villes
villes.plot(ax=ax, color="#e74c3c", markersize=150, zorder=5, edgecolor="white", linewidth=2)

# Annotations
for idx, row in villes.iterrows():
    ax.annotate(row["ville"], xy=(row.geometry.x, row.geometry.y),
                xytext=(8, 8), textcoords="offset points", 
                fontsize=11, fontweight="bold",
                bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.8))

ax.set_title("France: Villes et zones d'influence (50 km)", fontsize=14, fontweight="bold")
ax.set_xlim(-5, 10)
ax.set_ylim(41, 52)
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
plt.tight_layout()
plt.show()

In [ ]:
# 8.4 Personnalisation poussée
fig, ax = plt.subplots(figsize=(15, 8))

# Fond océan
ax.set_facecolor("#d4e5f7")

# Pays avec style personnalisé
world.plot(ax=ax, color="#2ecc71", edgecolor="#27ae60", linewidth=0.5)

# Villes capitales du dataset
cities.plot(ax=ax, color="#e74c3c", markersize=20, zorder=5)

ax.set_title("Carte du monde avec grandes villes", fontsize=16, fontweight="bold", pad=20)
ax.set_xlabel("Longitude", fontsize=12)
ax.set_ylabel("Latitude", fontsize=12)

# Grille
ax.grid(True, linestyle="--", alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# 8.5 Export haute résolution
fig, ax = plt.subplots(figsize=(15, 8))
ax.set_facecolor("#d4e5f7")
world.plot(column="continent", ax=ax, cmap="Set2", edgecolor="black", linewidth=0.3)
ax.set_title("Carte du monde", fontsize=14)
plt.tight_layout()

fig.savefig("carte_monde_hd.png", dpi=300, bbox_inches="tight")
print("✓ carte_monde_hd.png exportée (300 DPI)")
plt.show()

## Partie 9 — Exercices bonus

In [ ]:
# Bonus 1: Densité de population
world_density = world.copy()
world_density["density"] = world_density["pop_est"] / world_density["area_km2"]

fig, ax = plt.subplots(figsize=(15, 8))
world_density.plot(column="density", ax=ax, legend=True,
                   legend_kwds={"label": "Densité (hab/km²)", "shrink": 0.6},
                   cmap="YlOrRd", edgecolor="black", linewidth=0.3,
                   vmin=0, vmax=500)  # Limiter l'échelle

ax.set_title("Densité de population mondiale", fontsize=14)
plt.tight_layout()
plt.show()

print("=== Top 10 pays les plus denses ===")
display(world_density[["name", "density"]].sort_values("density", ascending=False).head(10))

In [ ]:
# Bonus 2: Plus proche voisin
villes_lambert = villes.to_crs(epsg=2154)

def find_nearest(gdf, target_idx):
    target = gdf.iloc[target_idx].geometry
    distances = gdf.geometry.distance(target)
    distances.iloc[target_idx] = np.inf  # Exclure soi-même
    nearest_idx = distances.idxmin()
    return gdf.iloc[nearest_idx]["ville"], distances[nearest_idx] / 1000

print("=== Plus proche voisin pour chaque ville ===")
for i, row in villes_lambert.iterrows():
    nearest, dist = find_nearest(villes_lambert, i)
    print(f"{row['ville']:12} → {nearest:12} ({dist:.0f} km)")

In [ ]:
# Bonus 3: Découpage avec bounding box (Europe de l'Ouest)
from shapely.geometry import box

bbox = box(-10, 35, 15, 60)  # Europe de l'Ouest
europe_ouest = world.clip(bbox)

fig, ax = plt.subplots(figsize=(10, 10))
europe_ouest.plot(ax=ax, cmap="Pastel1", edgecolor="black")
ax.set_title("Europe de l'Ouest (découpée par bounding box)")
plt.tight_layout()
plt.show()

In [ ]:
# Bonus 4: Agrégation par continent
continent_stats = world.dissolve(by="continent", aggfunc={"pop_est": "sum", "gdp_md_est": "sum"})

print("=== Statistiques par continent ===")
continent_stats["pop_millions"] = continent_stats["pop_est"] / 1e6
continent_stats["gdp_billions"] = continent_stats["gdp_md_est"] / 1e3
display(continent_stats[["pop_millions", "gdp_billions"]].round(1))

# Visualisation
fig, ax = plt.subplots(figsize=(15, 8))
continent_stats.plot(column="pop_est", ax=ax, legend=True, cmap="Blues", edgecolor="black")
ax.set_title("Population totale par continent")
plt.tight_layout()
plt.show()

In [ ]:
# Nettoyage des fichiers temporaires
import shutil

for f in ["europe.geojson", "europe.gpkg", "carte_monde_hd.png"]:
    if os.path.exists(f):
        os.remove(f)
        print(f"✓ {f} supprimé")

if os.path.exists("europe_shp"):
    shutil.rmtree("europe_shp")
    print("✓ europe_shp/ supprimé")